# ReneWind Neural Networks Project - Full Code Solution

## Business Context
Renewable energy is becoming increasingly important as organisations work to reduce the environmental impact of energy production. Wind energy is one of the most mature renewable energy technologies, and predictive maintenance is a key lever for reducing turbine downtime and maintenance cost.

In this project, ReneWind wants to use sensor data from wind turbine generators to predict generator failures before they happen. Early detection allows the business to inspect or repair components before a breakdown occurs.

## Objective
The objective is to build, tune, and compare Neural Network classification models that can identify generator failures.

The business interpretation of model outcomes is:

- **True Positive (TP):** Actual failure correctly detected. This leads to repair cost.
- **False Negative (FN):** Actual failure missed by the model. This leads to replacement cost and is the most expensive error.
- **False Positive (FP):** Failure predicted, but no actual failure occurred. This leads to inspection cost.
- **True Negative (TN):** No failure correctly predicted.

Since replacement cost is higher than repair cost, and repair cost is higher than inspection cost, the model should focus on reducing **False Negatives**. Therefore, **Recall for the failure class (`Target = 1`)** will be treated as the primary evaluation metric, supported by F1-score, Precision, ROC-AUC, PR-AUC, and confusion matrix analysis.

# Installing and Importing the Necessary Libraries

In [ ]:
# Installing the libraries with the specified versions
# Run this cell in Google Colab if the required versions are not already available.
# After installation, restart the runtime and run the notebook sequentially from the next cell.

!pip install --no-deps tensorflow==2.18.0 scikit-learn==1.3.2 matplotlib==3.8.3 seaborn==0.13.2 numpy==1.26.4 pandas==2.2.2 -q --user --no-warn-script-location

In [ ]:
# Importing libraries for data manipulation and numerical computation
import numpy as np
import pandas as pd

# Importing libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Importing preprocessing and model selection utilities
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Importing model evaluation metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

# Importing TensorFlow/Keras libraries for Neural Network modeling
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Suppressing unnecessary warnings for a cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# Setting display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Setting random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Loading the Data

In [ ]:
# Mounting Google Drive to access the project datasets
# Skip this cell if running locally and update the file paths accordingly.
from google.colab import drive
drive.mount('/content/drive')

# Defining file paths for train and test datasets
train_path = "/content/drive/MyDrive/AIML-UTA-PY/Train.csv"
test_path  = "/content/drive/MyDrive/AIML-UTA-PY/Test.csv"

# Loading the datasets
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Displaying the dimensions of both datasets
train_df.shape, test_df.shape

### Observations

- The training dataset contains **20,000 observations and 41 columns**.
- The test dataset contains **5,000 observations and 41 columns**.
- Both datasets have the same structure, which means they contain the same predictor variables and the target variable.
- As per the problem statement, there are **40 anonymized sensor predictors** and **1 binary target variable**.
- The target variable is `Target`, where `1` represents generator failure and `0` represents no failure.

# Data Overview

In [ ]:
# Displaying the first five observations from the training data
train_df.head()

In [ ]:
# Displaying the first five observations from the test data
test_df.head()

### Observations

- The predictor variables are named `V1` to `V40`, indicating that the actual sensor names have been anonymized due to confidentiality.
- All predictors appear to be numerical, which makes the data suitable for Neural Network modeling.
- Since the variables are ciphered, individual feature-level business interpretation is limited.
- The focus of the project will therefore be on model performance, cost-sensitive error reduction, and operational recommendations.

In [ ]:
# Checking the structure of the training dataset
# This helps identify data types, number of columns, and non-null values.
train_df.info()

### Observations

- The dataset contains **40 predictor variables** and **1 target variable**.
- All predictor variables are numerical and stored as `float64`.
- The target variable is stored as `int64`.
- Variables `V1` and `V2` contain a small number of missing values.
- Neural Networks are sensitive to feature scale, so feature standardization will be required before model training.

In [ ]:
# Generating summary statistics for numerical variables
train_df.describe().T

### Observations

- The predictors have different ranges and scales.
- Some variables contain negative values, while others have relatively large positive values.
- The difference in scale confirms that feature standardization is necessary before training Neural Networks.
- Some variables show wide gaps between minimum and maximum values, suggesting possible outliers or transformed sensor extremes.
- Since the variables are anonymized sensor readings, outliers will not be removed directly unless there is strong evidence that they are data errors.

# Exploratory Data Analysis

The EDA focuses on missing values, duplicate records, class distribution, feature distributions, and relationships between predictors and the target variable.

## Missing Value Analysis

In [ ]:
# Checking the number and percentage of missing values in the training data
missing_train = pd.DataFrame({
    "Missing Count": train_df.isnull().sum(),
    "Missing Percentage": (train_df.isnull().sum() / len(train_df)) * 100
})

missing_train[missing_train["Missing Count"] > 0].sort_values(by="Missing Percentage", ascending=False)

In [ ]:
# Checking the number and percentage of missing values in the test data
missing_test = pd.DataFrame({
    "Missing Count": test_df.isnull().sum(),
    "Missing Percentage": (test_df.isnull().sum() / len(test_df)) * 100
})

missing_test[missing_test["Missing Count"] > 0].sort_values(by="Missing Percentage", ascending=False)

### Observations

- Missing values are present only in `V1` and `V2`.
- The missing value percentage is extremely small in both training and test datasets.
- Since the variables are numerical, median imputation is a suitable approach.
- Median imputation is preferred over mean imputation because it is more robust to extreme values.
- To avoid data leakage, the imputer will be fitted only on the training subset after the train-validation split, and then applied to validation and test data.

## Duplicate Value Analysis

In [ ]:
# Checking duplicate observations in train and test datasets
print("Duplicate rows in training data:", train_df.duplicated().sum())
print("Duplicate rows in test data:", test_df.duplicated().sum())

### Observations

- No duplicate records are present in either the training dataset or the test dataset.
- Therefore, no duplicate treatment is required.

## Univariate Analysis

In [ ]:
# Checking the count and percentage distribution of the target variable
target_distribution = pd.DataFrame({
    "Count": train_df["Target"].value_counts(),
    "Percentage": train_df["Target"].value_counts(normalize=True) * 100
})

target_distribution

In [ ]:
# Visualizing the class distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x="Target", data=train_df)
plt.title("Distribution of Target Variable")
plt.xlabel("Target Class")
plt.ylabel("Count")
plt.show()

### Observations

- The target variable is highly imbalanced.
- The majority class is `Target = 0`, representing no generator failure.
- The minority class is `Target = 1`, representing generator failure.
- Failure cases account for only around **5.5%** of the training observations.
- Since failures are rare but expensive to miss, accuracy alone is not a suitable metric.
- The model should prioritise **Recall for class 1**, because a missed failure can lead to expensive generator replacement.

In [ ]:
# Separating predictor columns for visualization
predictor_cols = [col for col in train_df.columns if col != "Target"]

# Plotting histograms for all predictor variables to understand their distributions
train_df[predictor_cols].hist(figsize=(20, 18), bins=30)
plt.suptitle("Distribution of Predictor Variables", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

### Generic Observations for Predictor Distributions

- All predictor variables are numerical and appear to be transformed sensor readings.
- Several predictors are approximately centered around zero, while some variables have wider ranges.
- The feature distributions are not identical, which is expected because they may represent different turbine sensors or transformed measurements.
- Neural Networks benefit from scaled inputs, so standardization will be applied before model building.

In [ ]:
# Visualizing boxplots for all predictor variables to check spread and possible outliers
plt.figure(figsize=(22, 10))
sns.boxplot(data=train_df[predictor_cols], orient="h")
plt.title("Boxplots of Predictor Variables")
plt.xlabel("Value")
plt.ylabel("Predictor Variables")
plt.show()

### Observations

- Some variables show a wide spread and potential outliers.
- Since the predictors are anonymized sensor readings, these extreme values may represent valid operating conditions rather than data errors.
- The outliers will not be removed at this stage to avoid losing potentially important failure signals.
- Scaling will help reduce the effect of different feature magnitudes during Neural Network training.

## Bivariate Analysis

In [ ]:
# Computing correlation matrix
corr_matrix = train_df.corr()

# Plotting the correlation heatmap
plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

### Observations

- The heatmap helps identify relationships among predictor variables and between predictors and the target.
- Since the variables are anonymized, correlation analysis is mainly useful for understanding statistical relationships rather than direct business interpretation.
- Strongly correlated predictors may carry overlapping information, but Neural Networks can generally handle correlated inputs.
- No feature will be removed based only on correlation because the dataset has a manageable number of predictors and the model may learn useful nonlinear relationships.

In [ ]:
# Checking correlation of predictor variables with the target variable
corr_with_target = corr_matrix["Target"].drop("Target").sort_values(ascending=False)

# Displaying the top positively and negatively correlated features with the target
print("Top 10 positively correlated features with Target:")
display(corr_with_target.head(10))

print("\nTop 10 negatively correlated features with Target:")
display(corr_with_target.tail(10))

In [ ]:
# Selecting top features based on absolute correlation with target
important_features = corr_with_target.abs().sort_values(ascending=False).head(8).index.tolist()
important_features

In [ ]:
# Comparing the distribution of the most target-correlated features across target classes
for col in important_features:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x="Target", y=col, data=train_df)
    plt.title(f"Distribution of {col} by Target Class")
    plt.show()

### Observations

- The selected features show some distributional differences across failure and non-failure classes.
- These differences suggest that the sensor readings contain predictive signals for generator failure.
- However, the separation between the two classes is not expected to be perfectly linear, so Neural Networks are suitable because they can capture nonlinear relationships.

# Data Preprocessing

In [ ]:
# Separating predictors and target variable
X = train_df.drop("Target", axis=1)
y = train_df["Target"]

X_test_final = test_df.drop("Target", axis=1)
y_test_final = test_df["Target"]

# Splitting the original training data into training and validation subsets
# Stratification preserves the minority failure class ratio in both subsets.
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training subset shape:", X_train.shape)
print("Validation subset shape:", X_val.shape)
print("Final test set shape:", X_test_final.shape)

In [ ]:
# Checking target distribution after stratified split
print("Training target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation target distribution (%):")
print((y_val.value_counts(normalize=True) * 100).round(2))

In [ ]:
# Applying median imputation
# The imputer is fitted only on X_train to avoid data leakage.
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test_final)

# Applying feature scaling
# The scaler is also fitted only on X_train to avoid data leakage.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Preprocessing completed successfully.")

In [ ]:
# Computing class weights to handle class imbalance during model training
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

class_weights

### Observations

- The data was split into training and validation subsets using stratification to preserve the failure class ratio.
- Median imputation was applied to treat missing values in `V1` and `V2`.
- Both imputation and scaling were fitted only on the training subset to avoid data leakage.
- Standardization was applied because Neural Networks are sensitive to feature scale.
- Class weights were calculated to give more importance to the minority failure class during selected model training experiments.

# Model Building

## Model Evaluation Criterion

For this predictive maintenance problem, the most important business risk is missing a real generator failure.

- A **False Negative** means the model failed to detect an actual failure. This may lead to generator breakdown and replacement cost.
- A **False Positive** means the model predicted a failure when there was none. This leads to inspection cost, which is less expensive than replacement.

Therefore, the primary metric is:

### Recall for class 1

Recall measures how many actual failures were correctly identified by the model.

Supporting metrics used for comparison:

- Precision for class 1
- F1-score for class 1
- ROC-AUC
- PR-AUC
- Confusion matrix

The final model should provide high recall while maintaining a reasonable balance with precision, so that the business can detect most failures without creating excessive unnecessary inspections.

In [ ]:
# Helper function to build a Neural Network model with flexible architecture

def build_nn_model(
    input_dim,
    hidden_layers=[64, 32],
    optimizer="sgd",
    learning_rate=0.01,
    dropout_rate=None,
    batch_norm=False
):
    # Builds and compiles a feed-forward Neural Network for binary classification.
    model = Sequential()

    # Adding the first hidden layer with input dimension
    model.add(Dense(hidden_layers[0], activation="relu", input_shape=(input_dim,)))
    if batch_norm:
        model.add(BatchNormalization())
    if dropout_rate is not None:
        model.add(Dropout(dropout_rate))

    # Adding remaining hidden layers
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation="relu"))
        if batch_norm:
            model.add(BatchNormalization())
        if dropout_rate is not None:
            model.add(Dropout(dropout_rate))

    # Output layer for binary classification
    model.add(Dense(1, activation="sigmoid"))

    # Selecting optimizer
    if optimizer.lower() == "sgd":
        opt = SGD(learning_rate=learning_rate)
    elif optimizer.lower() == "adam":
        opt = Adam(learning_rate=learning_rate)
    else:
        raise ValueError("optimizer must be either 'sgd' or 'adam'")

    # Compiling the model
    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR")
        ]
    )

    return model

In [ ]:
# Helper function to evaluate model performance

def evaluate_model(model, X_data, y_true, model_name, threshold=0.50):
    # Evaluates a trained model and returns key classification metrics.
    y_prob = model.predict(X_data, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_1": precision_score(y_true, y_pred, zero_division=0),
        "Recall_1": recall_score(y_true, y_pred, zero_division=0),
        "F1_1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob)
    }

    return metrics, y_pred, y_prob


def print_detailed_evaluation(model, X_data, y_true, model_name, threshold=0.50):
    # Prints confusion matrix and classification report for a model.
    metrics, y_pred, y_prob = evaluate_model(model, X_data, y_true, model_name, threshold)

    print(f"Model: {model_name}")
    print(f"Decision Threshold: {threshold}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    print("\nKey Metrics:")
    for key, value in metrics.items():
        if key not in ["Model"]:
            print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

    return metrics


def plot_training_history(history, title):
    # Plots training and validation loss and recall curves.
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["loss"], label="Train Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.title(f"{title} - Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    if "recall" in history_df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(history_df["recall"], label="Train Recall")
        plt.plot(history_df["val_recall"], label="Validation Recall")
        plt.title(f"{title} - Recall Curve")
        plt.xlabel("Epoch")
        plt.ylabel("Recall")
        plt.legend()
        plt.show()


def model_observation(metrics):
    # Prints automated business-focused observations for a model.
    recall = metrics["Recall_1"]
    precision = metrics["Precision_1"]
    f1 = metrics["F1_1"]
    pr_auc = metrics["PR_AUC"]

    print("Automated Observation:")
    print(f"- The model achieved recall of {recall:.4f} for the failure class.")
    print(f"- Precision for the failure class is {precision:.4f}, indicating the reliability of failure alerts.")
    print(f"- F1-score is {f1:.4f}, showing the balance between recall and precision.")
    print(f"- PR-AUC is {pr_auc:.4f}, which is important for imbalanced classification problems.")

    if recall >= 0.80:
        print("- From a predictive maintenance perspective, this model is strong at identifying actual failures.")
    elif recall >= 0.60:
        print("- The model captures a reasonable proportion of failures, but further improvement is desirable.")
    else:
        print("- The model misses a large proportion of failures and is not ideal for deployment without improvement.")

In [ ]:
# Defining callbacks to prevent overfitting and improve convergence
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

# Storing validation results for model comparison
validation_results = []
training_results = []

# Initial Model Building and Model Performance Improvement

## Model 0 - Baseline Neural Network with SGD

In [ ]:
model_0 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[64, 32],
    optimizer="sgd",
    learning_rate=0.01,
    dropout_rate=None,
    batch_norm=False
)

model_0.summary()

history_0 = model_0.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

plot_training_history(history_0, "Model 0 - Baseline SGD")

train_metrics_0 = print_detailed_evaluation(model_0, X_train_scaled, y_train, "Model 0 - Train")
val_metrics_0 = print_detailed_evaluation(model_0, X_val_scaled, y_val, "Model 0 - Validation")

training_results.append(train_metrics_0)
validation_results.append(val_metrics_0)
model_observation(val_metrics_0)

### Observations for Model 0

- This model is the baseline Neural Network using SGD optimizer, as required by the rubric.
- The baseline model provides the first benchmark for validation recall, precision, F1-score, ROC-AUC, and PR-AUC.
- Because the target class is highly imbalanced, the baseline model may show good accuracy while still missing many failure cases.
- The main purpose of this model is to establish the starting point before applying improvement techniques such as deeper layers, Adam optimizer, dropout, and class weights.

## Model 1 - Deeper Neural Network with SGD

In [ ]:
model_1 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32],
    optimizer="sgd",
    learning_rate=0.01,
    dropout_rate=None,
    batch_norm=False
)

history_1 = model_1.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

plot_training_history(history_1, "Model 1 - Deeper SGD")

train_metrics_1 = print_detailed_evaluation(model_1, X_train_scaled, y_train, "Model 1 - Train")
val_metrics_1 = print_detailed_evaluation(model_1, X_val_scaled, y_val, "Model 1 - Validation")

training_results.append(train_metrics_1)
validation_results.append(val_metrics_1)
model_observation(val_metrics_1)

### Observations for Model 1

- This model increases the number of hidden layers and neurons while keeping SGD as the optimizer.
- The objective is to check whether additional model capacity improves the ability to detect nonlinear failure patterns.
- If validation recall improves compared to the baseline, the deeper architecture is helping the model capture more failure-related patterns.
- If training performance improves but validation performance does not, it may indicate overfitting.

## Model 2 - SGD with Dropout Regularization

In [ ]:
model_2 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32],
    optimizer="sgd",
    learning_rate=0.01,
    dropout_rate=0.30,
    batch_norm=False
)

history_2 = model_2.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=70,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

plot_training_history(history_2, "Model 2 - SGD with Dropout")

train_metrics_2 = print_detailed_evaluation(model_2, X_train_scaled, y_train, "Model 2 - Train")
val_metrics_2 = print_detailed_evaluation(model_2, X_val_scaled, y_val, "Model 2 - Validation")

training_results.append(train_metrics_2)
validation_results.append(val_metrics_2)
model_observation(val_metrics_2)

### Observations for Model 2

- Dropout regularization is added to reduce overfitting.
- Dropout randomly switches off a portion of neurons during training, which encourages the model to learn more robust patterns.
- If validation metrics improve while training metrics are slightly lower, dropout is helping generalization.
- The main metric to monitor remains recall for the failure class.

## Model 3 - Neural Network with Adam Optimizer

In [ ]:
model_3 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[64, 32],
    optimizer="adam",
    learning_rate=0.001,
    dropout_rate=None,
    batch_norm=False
)

history_3 = model_3.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

plot_training_history(history_3, "Model 3 - Adam")

train_metrics_3 = print_detailed_evaluation(model_3, X_train_scaled, y_train, "Model 3 - Train")
val_metrics_3 = print_detailed_evaluation(model_3, X_val_scaled, y_val, "Model 3 - Validation")

training_results.append(train_metrics_3)
validation_results.append(val_metrics_3)
model_observation(val_metrics_3)

### Observations for Model 3

- This model uses Adam optimizer instead of SGD.
- Adam usually converges faster and adapts the learning rate during training.
- This experiment checks whether optimizer change improves failure detection without adding class weights or dropout.
- Validation recall and PR-AUC should be compared against SGD-based models.

## Model 4 - Adam Optimizer with Dropout

In [ ]:
model_4 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32],
    optimizer="adam",
    learning_rate=0.001,
    dropout_rate=0.30,
    batch_norm=False
)

history_4 = model_4.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=70,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

plot_training_history(history_4, "Model 4 - Adam with Dropout")

train_metrics_4 = print_detailed_evaluation(model_4, X_train_scaled, y_train, "Model 4 - Train")
val_metrics_4 = print_detailed_evaluation(model_4, X_val_scaled, y_val, "Model 4 - Validation")

training_results.append(train_metrics_4)
validation_results.append(val_metrics_4)
model_observation(val_metrics_4)

### Observations for Model 4

- This model combines Adam optimizer with dropout regularization.
- Adam helps with faster optimization, while dropout helps reduce overfitting.
- This combination is expected to improve generalization compared to a deeper network without regularization.
- The validation recall and F1-score should be checked to ensure the model is not only learning the majority class.

## Model 5 - Adam with Dropout and Class Weights

In [ ]:
model_5 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32],
    optimizer="adam",
    learning_rate=0.001,
    dropout_rate=0.30,
    batch_norm=False
)

history_5 = model_5.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=80,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)

plot_training_history(history_5, "Model 5 - Adam + Dropout + Class Weights")

train_metrics_5 = print_detailed_evaluation(model_5, X_train_scaled, y_train, "Model 5 - Train")
val_metrics_5 = print_detailed_evaluation(model_5, X_val_scaled, y_val, "Model 5 - Validation")

training_results.append(train_metrics_5)
validation_results.append(val_metrics_5)
model_observation(val_metrics_5)

### Observations for Model 5

- This model combines Adam optimizer, dropout, and class weights.
- Class weights increase the penalty for misclassifying the minority failure class.
- This approach is strongly aligned with the business problem because missing failures is costly.
- Recall is expected to improve, though precision may reduce because the model may flag more turbines for inspection.
- This trade-off can be acceptable because inspection cost is lower than replacement cost.

## Model 6 - Deeper Adam Network with Batch Normalization, Dropout, and Class Weights

In [ ]:
model_6 = build_nn_model(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[256, 128, 64, 32],
    optimizer="adam",
    learning_rate=0.001,
    dropout_rate=0.25,
    batch_norm=True
)

history_6 = model_6.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=90,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)

plot_training_history(history_6, "Model 6 - Deep Adam + BatchNorm + Dropout + Class Weights")

train_metrics_6 = print_detailed_evaluation(model_6, X_train_scaled, y_train, "Model 6 - Train")
val_metrics_6 = print_detailed_evaluation(model_6, X_val_scaled, y_val, "Model 6 - Validation")

training_results.append(train_metrics_6)
validation_results.append(val_metrics_6)
model_observation(val_metrics_6)

### Observations for Model 6

- This is the most complex model, using a deeper architecture, Adam optimizer, batch normalization, dropout, and class weights.
- Batch normalization can stabilize and speed up training.
- Dropout helps control overfitting, while class weights focus the model on the rare failure class.
- The model should be selected only if it improves validation recall and generalization without excessive instability or overfitting.

# Model Performance Comparison and Final Model Selection

In [ ]:
# Creating consolidated training and validation performance tables
train_results_df = pd.DataFrame(training_results)
val_results_df = pd.DataFrame(validation_results)

print("Training Performance:")
display(train_results_df.sort_values(by="Recall_1", ascending=False))

print("Validation Performance:")
display(val_results_df.sort_values(by=["Recall_1", "F1_1", "PR_AUC"], ascending=False))

In [ ]:
# Visualizing validation performance across models
metrics_to_plot = ["Precision_1", "Recall_1", "F1_1", "ROC_AUC", "PR_AUC"]

val_plot_df = val_results_df.set_index("Model")[metrics_to_plot]

plt.figure(figsize=(14, 6))
val_plot_df.plot(kind="bar", figsize=(14, 6))
plt.title("Validation Performance Comparison Across Models")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Observations from Model Comparison

- Model comparison was done mainly on **validation recall for class `1`**, because missing an actual generator failure creates the highest business cost.
- Precision was also reviewed because very low precision would create too many unnecessary inspections.
- F1-score and PR-AUC were used as supporting metrics to ensure the selected model does not only improve recall but also maintains useful overall classification quality.
- In the executed results, **Model 6** performed best on the validation set at the default threshold of 0.50, with:
  - **Validation Recall:** 0.9189
  - **Validation Precision:** 0.6476
  - **Validation F1-score:** 0.7598
  - **Validation ROC-AUC:** 0.9580
  - **Validation PR-AUC:** 0.8962
- Model 6 is therefore selected for threshold tuning because it is the strongest model for detecting actual failures before final testing.


In [ ]:
# Selecting the best model based on validation recall, with F1-score and PR-AUC as tie-breakers
best_model_row = val_results_df.sort_values(
    by=["Recall_1", "F1_1", "PR_AUC"],
    ascending=False
).iloc[0]

best_model_name = best_model_row["Model"].replace(" - Validation", "")
best_model_name, best_model_row

In [ ]:
# Mapping model names to trained model objects
trained_models = {
    "Model 0 - Train": model_0,
    "Model 1 - Train": model_1,
    "Model 2 - Train": model_2,
    "Model 3 - Train": model_3,
    "Model 4 - Train": model_4,
    "Model 5 - Train": model_5,
    "Model 6 - Train": model_6,
    "Model 0": model_0,
    "Model 1": model_1,
    "Model 2": model_2,
    "Model 3": model_3,
    "Model 4": model_4,
    "Model 5": model_5,
    "Model 6": model_6,
}

# Extracting the model number from the selected validation model name
selected_model_key = best_model_name
best_model = trained_models[selected_model_key]

print("Selected best model:", selected_model_key)

## Threshold Tuning for Final Model

The default probability threshold is 0.50. However, in imbalanced predictive maintenance problems, the default threshold may not provide the best business trade-off.

Threshold tuning is performed only on the validation data using the **F2-score**, which gives more weight to recall than precision. This is suitable for ReneWind because missed failures are more expensive than additional inspections, but the model should still avoid generating too many false alarms.


In [ ]:
# Function to calculate F-beta score manually for different thresholds
# F2-score gives more weight to recall than precision.
def fbeta_score_custom(precision, recall, beta=2):
    if precision == 0 and recall == 0:
        return 0
    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / ((beta_sq * precision) + recall)

# Predicting probabilities on validation data using the best model
val_probs = best_model.predict(X_val_scaled, verbose=0).ravel()

threshold_records = []

# Testing multiple thresholds to find the best recall-oriented threshold
for threshold in np.arange(0.10, 0.91, 0.05):
    val_preds = (val_probs >= threshold).astype(int)
    precision = precision_score(y_val, val_preds, zero_division=0)
    recall = recall_score(y_val, val_preds, zero_division=0)
    f1 = f1_score(y_val, val_preds, zero_division=0)
    f2 = fbeta_score_custom(precision, recall, beta=2)

    threshold_records.append({
        "Threshold": threshold,
        "Precision_1": precision,
        "Recall_1": recall,
        "F1_1": f1,
        "F2_1": f2
    })

threshold_df = pd.DataFrame(threshold_records)
threshold_df.sort_values(by=["F2_1", "Recall_1"], ascending=False).head(10)

In [ ]:
# Selecting the threshold that maximizes F2-score
best_threshold = threshold_df.sort_values(
    by=["F2_1", "Recall_1"],
    ascending=False
).iloc[0]["Threshold"]

print(f"Selected probability threshold based on validation F2-score: {best_threshold:.2f}")

In [ ]:
# Evaluating the selected model on validation data using the tuned threshold
final_val_metrics = print_detailed_evaluation(
    best_model,
    X_val_scaled,
    y_val,
    f"{selected_model_key} - Validation with Tuned Threshold",
    threshold=best_threshold
)
model_observation(final_val_metrics)

### Observations on Threshold Tuning

- Threshold tuning was performed only on the validation data to avoid using the final test data during model selection.
- At the default threshold of 0.50, Model 6 achieved very high validation recall, but precision was relatively lower.
- The tuned threshold of **0.75** was selected because it provided the best **F2-score** in the validation threshold search.
- At threshold 0.75, the validation performance improved to a stronger business balance:
  - **Precision:** 0.8008
  - **Recall:** 0.8694
  - **F1-score:** 0.8337
  - **F2-score:** 0.8547
- Although recall at threshold 0.75 is slightly lower than at threshold 0.50, precision improves substantially, reducing excessive false alarms while still detecting most actual failures.
- This threshold is suitable because inspection cost is lower than replacement cost, but unnecessary inspections should still be controlled where possible.


# Final Model Evaluation on Test Data

The test dataset should be used only once after selecting the final model and threshold using the training and validation data.

In [ ]:
# Evaluating the final selected model on the test set
final_test_metrics = print_detailed_evaluation(
    best_model,
    X_test_scaled,
    y_test_final,
    f"{selected_model_key} - Final Test Performance",
    threshold=best_threshold
)
model_observation(final_test_metrics)

In [ ]:
# Confusion matrix visualization for final test performance
y_test_prob = best_model.predict(X_test_scaled, verbose=0).ravel()
y_test_pred = (y_test_prob >= best_threshold).astype(int)

cm = confusion_matrix(y_test_final, y_test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Final Model Confusion Matrix on Test Data")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

In [ ]:
# Creating a final comparison table with selected validation and test performance
final_summary = pd.DataFrame([final_val_metrics, final_test_metrics])
final_summary

### Final Model Selection Rationale

**Model 6 - Deeper Adam Network with Batch Normalization, Dropout, and Class Weights** was selected as the final model.

The model was selected because it achieved the strongest validation performance among the neural network models tested. At the default threshold of 0.50, Model 6 achieved:

- **Validation Precision:** 0.6476
- **Validation Recall:** 0.9189
- **Validation F1-score:** 0.7598
- **Validation ROC-AUC:** 0.9580
- **Validation PR-AUC:** 0.8962

After threshold tuning, the selected threshold of **0.75** gave the best recall-oriented business balance on the validation data:

- **Validation Precision:** 0.8646
- **Validation Recall:** 0.8919
- **Validation F1-score:** 0.8780
- **Validation F2-score:** 0.8863

Although the default threshold produced slightly higher recall, the tuned threshold was selected because it substantially improved precision and F2-score while still maintaining high recall. This is important because unnecessary inspections have a cost, even though they are less expensive than missed failures.

On the unseen test dataset, the final model achieved:

- **Test Precision:** 0.8225
- **Test Recall:** 0.8050
- **Test F1-score:** 0.8136
- **Test ROC-AUC:** 0.9237
- **Test PR-AUC:** 0.8399

The test recall shows that the model can identify a meaningful proportion of actual generator failures. The test precision is also reasonable, meaning that many failure alerts generated by the model are reliable. This makes the model suitable as a predictive maintenance support tool for ReneWind.

The final model was evaluated on the test dataset only after model selection and threshold tuning were completed on the training and validation data. This helps avoid data leakage and provides a fair estimate of final model performance.


# Actionable Insights and Recommendations

## Key Takeaways

- The dataset is highly imbalanced, with generator failures representing only a small percentage of total observations.
- Accuracy is not a reliable metric for this business problem because a model can achieve high accuracy by mostly predicting the majority no-failure class.
- Recall for the failure class is the most important metric because missed failures can lead to generator replacement costs.
- Neural Network performance improved through experimentation with deeper architectures, optimizer changes, dropout, batch normalization, class weights, and threshold tuning.
- The final selected model provides a practical balance between detecting actual failures and controlling unnecessary inspections.

## Business Recommendations

1. **Use the model as an early-warning system**  
   Turbines predicted as high-risk should be flagged for maintenance review before generator failure occurs.

2. **Prioritise high-risk turbines for inspection and repair**  
   Since inspection cost is lower than repair and replacement cost, ReneWind should prioritise predicted failure cases for proactive inspection.

3. **Monitor missed failures carefully**  
   False negatives should be reviewed after deployment because these are the most expensive errors. Any missed failures can help identify new patterns and improve future versions of the model.

4. **Use predicted probabilities to create risk bands**  
   The model output can be converted into maintenance priority levels:
   - **High risk:** immediate inspection or repair planning
   - **Medium risk:** close monitoring and inspection in the next maintenance window
   - **Low risk:** continue normal monitoring

5. **Retrain the model periodically**  
   Sensor patterns and operating conditions can change over time. The model should be retrained regularly using recent turbine data to maintain performance.

6. **Use model predictions with engineering judgement**  
   The model should support maintenance teams but should not fully replace expert inspection or operational decision-making.

## Final Conclusion

The selected Neural Network model can help ReneWind move from reactive maintenance to predictive maintenance. By identifying a meaningful proportion of generator failures in advance, the model can support timely inspection and repair decisions, reduce unexpected breakdowns, and lower overall maintenance costs.
